# 00 · Integración y auditoría inicial
**StreamView Analytics · EP5 · reproducible**

**Propósito:** integrar los dos archivos oficiales en memoria; no generar CSV redundantes. Los datos originales permanecen en `data/raw/` sin modificaciones. El resultado de este notebook es exploratorio y **no se exporta**. La exportación se hace únicamente en el notebook 02.

**Grano de origen:** una fila por registro suministrado; no necesariamente un título único. Evitar afirmar que hay 32.000 obras únicas hasta resolver conflictos de identificadores.

In [3]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src' / 'pipeline.py').exists() else Path.cwd().parent
assert (ROOT / 'src' / 'pipeline.py').exists(), 'Ejecuta Jupyter desde la raíz del proyecto o notebooks/'
sys.path.insert(0,str(ROOT))
import pandas as pd, numpy as np
from IPython.display import display
from src.pipeline import load_sources, clean, audit, kpis, looker_table, OUT


## 1. Lectura sin alterar originales

In [4]:
raw=load_sources()
print("Registros recibidos:",len(raw))
print(raw.groupby("type",dropna=False).size())
display(raw.head(3))

Registros recibidos: 32000
type
Movie      16000
TV Show    16000
dtype: int64


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000.0,752600867.0
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000.0,839030630.0
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000.0,954305868.0


## 2. Auditoría de faltantes y tipos
No confundir ausencia estructural (presupuesto de series) con errores; los nulos no deben imputarse artificialmente.

In [5]:
display(audit(raw).sort_values("porcentaje",ascending=False).reset_index(drop=True))
print(raw.dtypes.to_string())

,campo,faltantes,porcentaje
0,budget,16000,50.00
1,duration,16000,50.00
2,revenue,16000,50.00
3,director,11097,34.68
4,description,3338,10.43
5,country,2263,7.07
6,cast,1361,4.25
7,genres,1081,3.38
8,type,0,0.00
9,show_id,0,0.00


show_id           int64
type             object
title            object
director         object
cast             object
country          object
date_added       object
release_year      int64
rating          float64
duration         object
genres           object
language         object
description      object
popularity      float64
vote_count        int64
vote_average    float64
budget          float64
revenue         float64


## 3. Conflictos de clave
`show_id` debe evaluarse junto con `type`. Las nueve repeticiones de series corresponden a los mismos títulos y años pero difieren en popularidad: **no se deben convertir automáticamente en títulos adicionales**.

In [6]:
conf=raw[raw.duplicated(["type","show_id"],keep=False)].sort_values(["type","show_id"])
print("Claves (type,show_id) repetidas:",conf.groupby(["type","show_id"]).ngroups)
display(conf[["type","show_id","title","release_year","popularity","vote_count"]])

Claves (type,show_id) repetidas: 9


,type,show_id,title,release_year,popularity,vote_count
31539,TV Show,252630,Joe Lycett’s United States of Birmingham,2025,11.550,0
31540,TV Show,252630,Joe Lycett’s United States of Birmingham,2025,11.531,0
31358,TV Show,278867,Ensemble,2025,11.044,1
31360,TV Show,278867,Ensemble,2025,18.407,1
31398,TV Show,279457,Ghassan & Mina,2025,9.631,0
31400,TV Show,279457,Ghassan & Mina,2025,16.051,0
31479,TV Show,279739,Settle Down,2025,15.215,0
31480,TV Show,279739,Settle Down,2025,13.135,0
31519,TV Show,280651,Master in 3 Months: Edo Period,2025,16.794,0
31520,TV Show,280651,Master in 3 Months: Edo Period,2025,11.996,0


## 4. Control de cobertura temporal y atributos cuestionables
Si el muestreo es perfectamente balanceado, las barras anuales **no representan crecimiento orgánico**. La columna `rating` se comporta como `vote_average`, no como clasificación etaria.

In [8]:
display(raw.groupby(["release_year","type"]).size().unstack(fill_value=0))
print("Proporción de coincidencia rating y vote_average:",(pd.to_numeric(raw.rating,errors="coerce").eq(pd.to_numeric(raw.vote_average,errors="coerce"))).mean())
print("Coincidencia año date_added y release_year:",(pd.to_datetime(raw.date_added,errors="coerce").dt.year.eq(raw.release_year)).mean())

type,Movie,TV Show
release_year,,
2010,1000,1000
2011,1000,1000
2012,1000,1000
2013,1000,1000
2014,1000,1000
2015,1000,1000
2016,1000,1000
2017,1000,1000
2018,1000,1000


Proporción de coincidencia rating y vote_average: 1.0
Coincidencia año date_added y release_year: 1.0


## Hallazgos y decisiones de integración
1. Se recibieron **32.000 registros**, 16.000 por tipo. Es una muestra balanceada; el 50/50 no demuestra una estrategia del catálogo real.
2. Se identificaron **9 claves de series repetidas**; `01` documenta una regla de consolidación y reduce el número de obras únicas a 31.991.
3. `rating` es numérica y duplica `vote_average`; no sirve para segmentación etaria.
4. Las fechas de incorporación no deben tratarse como evidencia independiente de entradas al catálogo: su año coincide con el de estreno.
5. No se crean archivos derivados aquí; el pipeline está centralizado en `src/pipeline.py`.

**Limitación:** la extracción académica no incluye retención, reproducciones ni suscripciones. Por tanto, no se pueden inferir fidelidad, engagement o causalidad.